# HanuAI ML Assessment — Task 1: Review Sentiment and Aspect Analysis

## Dataset status and important limitation
This notebook analyses 50 review records selected from the supplied `DatafinitiElectronicsProductData(1).csv` dataset. The records have Best Buy review URLs listed in the `reviews.sourceURLs` field and are for **Microsoft Surface Pro 4 Type Cover with Fingerprint ID**.

**This is not a live scrape.** The supplied dataset does not contain the originally configured DJI Avata 2 product, so these records do not satisfy a requirement specifically tied to that DJI product. They are genuine-looking source dataset records with Best Buy review URL evidence, but source attribution is based on the dataset's URL metadata. Confirm with the assessment instructions/team whether this alternative dataset is acceptable.

The 50-row file `bestbuy_reviews_50.csv` is included alongside this notebook. The analysis uses TextBlob polarity (not VADER compound scores) and keyword-based aspect tagging.

In [ ]:
%pip install pandas textblob

In [ ]:
import hashlib
import pandas as pd
from textblob import TextBlob
from IPython.display import display

INPUT_CSV = "bestbuy_reviews_50.csv"
OUTPUT_CSV = "task1_reviews_with_sentiment.csv"
MIN_REVIEWS = 50

ASPECT_KEYWORDS = {
    "Battery": ["battery", "charge", "charging", "flight time"],
    "Performance": ["performance", "fast", "speed", "lag", "works", "work"],
    "Design": ["design", "look", "build", "size", "weight", "light", "thin"],
    "Price": ["price", "expensive", "cheap", "value", "cost"],
    "Quality": ["quality", "durable", "broken", "defect", "reliable", "solid", "sturdy"],
    "Customer Support": ["support", "service", "refund", "replacement"],
    "Delivery": ["delivery", "shipping", "package", "arrived"],
    "Camera": ["camera", "video", "image", "photo", "resolution"],
    "App & Connectivity": ["app", "connection", "connectivity", "bluetooth", "wifi", "fingerprint", "recognition"],
}

def classify_sentiment(text):
    score = float(TextBlob(str(text)).sentiment.polarity)
    if score >= 0.05:
        label = "Positive"
    elif score <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    return label, score

def tag_aspects(text):
    text = str(text).lower()
    found = [
        aspect for aspect, keywords in ASPECT_KEYWORDS.items()
        if any(keyword in text for keyword in keywords)
    ]
    return ", ".join(found) if found else "Other"

print("Setup complete")

In [ ]:
reviews = pd.read_csv(INPUT_CSV)
reviews.columns = reviews.columns.str.strip()

required = {"Review", "Rating"}
missing = required - set(reviews.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

for col in ["Title", "Date", "Reviewer", "Source", "Product_Name", "Brand", "Source_URLs"]:
    if col not in reviews.columns:
        reviews[col] = ""

reviews["Review"] = reviews["Review"].fillna("").astype(str).str.strip()
reviews["Rating"] = pd.to_numeric(reviews["Rating"], errors="coerce")
reviews = reviews[(reviews["Review"] != "") & reviews["Rating"].between(1, 5)].copy()
reviews["Primary_Key"] = reviews["Review"].str.lower().apply(
    lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest()[:12].upper()
)
reviews = reviews.drop_duplicates(subset=["Primary_Key"]).reset_index(drop=True)

if len(reviews) < MIN_REVIEWS:
    raise ValueError(f"Only {len(reviews)} valid unique reviews; {MIN_REVIEWS} required.")

reviews["Review_Clean"] = reviews["Review"].str.lower()
combined_text = reviews["Title"].fillna("").astype(str) + " " + reviews["Review"]
sentiment_results = combined_text.apply(classify_sentiment)
reviews["Overall_Sentiment"] = sentiment_results.apply(lambda x: x[0])
reviews["Polarity_Score"] = sentiment_results.apply(lambda x: round(x[1], 4))
reviews["Aspects"] = combined_text.apply(tag_aspects)
reviews["Sentiment_Tags"] = reviews["Aspects"]

reviews.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Rows processed:", len(reviews))
print("Output saved:", OUTPUT_CSV)
display(reviews[["Title", "Rating", "Review", "Overall_Sentiment", "Polarity_Score", "Aspects"]].head(10))

In [ ]:
print("Sentiment distribution:")
sentiment_summary = reviews["Overall_Sentiment"].value_counts().rename_axis("Sentiment").to_frame("Count")
sentiment_summary["Percent"] = (sentiment_summary["Count"] / len(reviews) * 100).round(1)
display(sentiment_summary)

print("Aspect distribution (reviews can have multiple aspects):")
aspect_summary = reviews["Aspects"].str.split(", ").explode().value_counts().rename_axis("Aspect").to_frame("Review_Count")
display(aspect_summary)

print("Source distribution:")
display(reviews["Source"].value_counts().to_frame("Rows"))

assert len(reviews) >= MIN_REVIEWS
assert reviews["Review"].notna().all()
assert reviews["Review"].str.strip().ne("").all()
print("Validation passed.")

## Interpretation and limitations
- This run processes 50 records from the supplied dataset, not reviews scraped live during this run.
- The dataset's review-source URL metadata includes Best Buy review URLs, but this is not the same as proving each row was collected from the exact target product page.
- The selected product is Microsoft Surface Pro 4 Type Cover, not DJI Avata 2. If the task requires the DJI product specifically, a matching approved dataset is still needed.
- TextBlob polarity is a lightweight baseline; keyword aspect tagging can miss context, sarcasm, and synonyms.
- Do not describe these results as sentiment about the DJI Avata 2 product.